In [29]:
#Data 불러오기
import pandas as pd
train=pd.read_csv("../Data/train_20k.csv", header=None)
test=pd.read_csv("../Data/test_1k.csv", header=None)

In [30]:
#train의 data와 target 구분하기
train_data=train.iloc[:,1:]
train_target=train.iloc[:,:1]

In [31]:
#test의 data와 target 구분하기
test_data=test.iloc[:,1:]
test_target=test.iloc[:,:1]

#Data들을 Tensor로 변환

In [32]:
import torch
train_input=torch.tensor(train_data.values)#array(numpy)
train_target=torch.tensor(train_target.values)

test_input=torch.tensor(test_data.values)#array(numpy)
test_target=torch.tensor(test_target.values)

print(train_input.data.shape)
print(train_target.data.shape)
print(test_input.data.shape)
print(test_target.data.shape)
#numpy array=>tensor

torch.Size([20001, 784])
torch.Size([20001, 1])
torch.Size([1001, 784])
torch.Size([1001, 1])


----
#Data들의 차원 변경

In [ ]:
train_input= train_input.reshape(-1,28,28)
train_target= train_target.reshape(-1,)
test_input= test_input.reshape(-1,28,28)
test_target= test_target.reshape(-1,)

print(train_input.data.shape)
print(train_target.data.shape)
print(test_input.data.shape)
print(test_target.data.shape)

torch.Size([20001, 28, 28])
torch.Size([20001])
torch.Size([1001, 28, 28])
torch.Size([1001])


In [34]:
#데이터 표준화 및 2차원 행렬
train_scaled=(train_input/255.0).reshape(-1,28,28)
test_scaled=(test_input/255.0).reshape(-1,28,28)

print(train_scaled.shape)
print(test_scaled.shape)

torch.Size([20001, 28, 28])
torch.Size([1001, 28, 28])


In [ ]:
#모델 만들기
import torch
import torch.nn as nn#뉴멀 네트워크?
import torch.optim as optim#optimize
from torch.utils.data import TensorDataset,DataLoader#셔플링
#Train과 Valid
from sklearn.model_selection import train_test_split
train_scaled, val_scaled,train_target,val_target=train_test_split(
    train_scaled,train_target,test_size=0.2,random_state=42
)

In [36]:
#dataset과 dataloader(그대로 사용하지 못함) 생성
batch_size=32#mini batch#묶어서 async로 돌린다.
train_dataset=TensorDataset(train_scaled,train_target)
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)#머신러닝 crossvalidate
val_dataset=TensorDataset(val_scaled,val_target)
val_loader=DataLoader(val_dataset,batch_size=batch_size)#검증 셔플 필요없음 학습시에만 에포크, 셔플

#모델정의
:입력층->은닉층(활성화 함수)->출력층으로 구분

In [37]:
# from turtle import forward

# class NeuralNetwork(nn.Module):#상속(initialize)생성시
#     def __init__(self):#설정을 해줘야함
#         super(NeuralNetwork, self).__init__()
#         self.flatten=nn.Flatten()
#         self.fc1=nn.Linear(28*28,512)#임의의숫자
#         self.relu=nn.ReLU()
#         self.relu=nn.Linear(512,10)#linear512=>10:target의 수
#         self.softmax=nn.Softmax(dim=1)#정의 출력층/차원:1개
        
#     def forward(self,x):
#         x=self.flatten(x)
#         x=self.fc1(x)
#         x=self.relu(x)
#         x=self.fc1(x)
#         x=self.softmax(x)
#         return x#정방향 변수 정의

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 512)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512,10)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x

In [39]:
device=torch.device("mps" if torch.backends.mps.is_available() else 'cpu')
print(device)
# model.to(device)#summary

mps


In [40]:
#모델 인스턴스
model=NeuralNetwork().to(device)

In [41]:
#손실함수와 옵티마이저
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [42]:
#학습함수(한번 만들고 복사):순서중요
def train(model,train_loader,criterion,optimizer,device):
    model.train()#dropout 포함됨
    total_loss=0
    for inputs,targets in train_loader:
        inputs, targets =inputs.to(device), targets.to(device)#gpu에 옮겨놔야 계산을 해줌<<<<<<<<
        optimizer.zero_grad()#기울기 0:gradient 초기화
        outputs=model(inputs)#정방향
        loss=criterion(outputs,targets)#손실함수
        loss.backward()#역전파
        optimizer.step()#파라미터/가중치 바꿔줌
    return loss.item()

In [43]:
#평가함수
def evaluate(model, val_loader,criterion,device):
    model.eval()#llm:정방향:기울기?미분 바꿈
    total_loss=0#전체 손실 합계
    correct=0#정확하게 예측한 샘플수
    total=0#전체 샘플수
    with torch.no_grad():
        #with file close 안해도됨
        for inputs, targets in val_loader:
            inputs,targets= inputs.to(device),targets.to(device)
            outputs=model(inputs)
            loss=criterion(outputs,targets)
            total_loss+=loss.item()
            _, predicted= outputs.max(1)#선형회귀
            total+=targets.size(0)
            correct+=predicted.eq(targets).sum().item()

    return total_loss/len(val_loader),correct/total #평균손실, 정확도        


In [ ]:
#예측함수
def predict(model,data_loader,device):
    model.eval()
    predictions=[]#for문 하나짜리
    with torch.no_grad():
        for inputs, _ in data_loader:
            inputs = inputs.to(device)
            outputs=model(inputs)
            _,predicted=outputs.max(1)
            predictions.extend(predicted.cpu().numpy())
            return predictions

---
#학습 및 평가

In [45]:
model.to(device)#summary

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=512, out_features=10, bias=True)
)

In [46]:
#훈련하기 epochs for문
num_epochs=100
for epoch in range(num_epochs):
    train_loss=train(model, train_loader,criterion,optimizer,device)#loss만 return
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss:{train_loss:.4f}")

Epoch [1/100], Loss:0.1323
Epoch [2/100], Loss:0.2076
Epoch [3/100], Loss:0.0824
Epoch [4/100], Loss:0.0842
Epoch [5/100], Loss:0.0190
Epoch [6/100], Loss:0.0535
Epoch [7/100], Loss:0.0123
Epoch [8/100], Loss:0.0008
Epoch [9/100], Loss:0.0329
Epoch [10/100], Loss:0.0047
Epoch [11/100], Loss:0.0028
Epoch [12/100], Loss:0.0032
Epoch [13/100], Loss:0.0313
Epoch [14/100], Loss:0.0097
Epoch [15/100], Loss:0.0005
Epoch [16/100], Loss:0.0108
Epoch [17/100], Loss:0.0053
Epoch [18/100], Loss:0.0005
Epoch [19/100], Loss:0.0025
Epoch [20/100], Loss:0.0012
Epoch [21/100], Loss:0.0001
Epoch [22/100], Loss:0.0000
Epoch [23/100], Loss:0.0002
Epoch [24/100], Loss:0.0000
Epoch [25/100], Loss:0.0002
Epoch [26/100], Loss:0.0001
Epoch [27/100], Loss:0.0001
Epoch [28/100], Loss:0.0000
Epoch [29/100], Loss:0.0000
Epoch [30/100], Loss:0.0000
Epoch [31/100], Loss:0.0000
Epoch [32/100], Loss:0.0000
Epoch [33/100], Loss:0.0000
Epoch [34/100], Loss:0.0000
Epoch [35/100], Loss:0.0000
Epoch [36/100], Loss:0.0000
E

In [48]:
#훈련평가
train_loss, train_accuracy = evaluate(model, train_loader, criterion, device)
print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")

#검증평가
val_loss,val_accuracy=evaluate(model,val_loader,criterion,device)
print(f"Loss:{val_loss},Accuracy:{val_accuracy}")

Train Loss: 0.0000, Train Accuracy: 1.0000
Loss:0.2992012089339879,Accuracy:0.9720069982504373


In [49]:
#일반화 평가
test_dataset=TensorDataset(test_scaled, test_target)
test_loader=DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

#평가하기
test_loss,test_accuracy=evaluate(model,test_loader,criterion,device)
print(f"Loss:{test_loss},Accuracy:{test_accuracy}")

Loss:0.182654363513524,Accuracy:0.977022977022977


In [50]:
#예측
predictions=predict(model, test_loader,device)

In [51]:
#결과
print(test_target[:10])
print(predictions[:10])

tensor([7, 2, 1, 0, 4, 1, 4, 9, 5, 9])
[np.int64(6), np.int64(6), np.int64(1), np.int64(8), np.int64(9), np.int64(7), np.int64(7), np.int64(6), np.int64(5), np.int64(6)]


---
이미지 불러와서 predict해보기

In [52]:
from PIL import Image

In [53]:
#Image 불러오기
img=Image.open("../Data/mnist_test_3.jpg")
img

In [54]:
import numpy as np

In [56]:
#img=>numpy array
imgArray=np.array(img)
imgArray

array([[  0,   0,   0,   0,   0,   0,   0,   0,   3,   0,   0,  15,   6,
          0,   8,   0,   9,   0,   7,   0,   0,   0,   1,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,  11,   2,   0,   0,
          0,  18,   0,   0,   0,   0,   0,   0,   0,   0,   5,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  16,  19,
          0,   0,   5,   2,  18,   0,  18,   3,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,  13,   0,   0,   4,
          0,   0,   0,   6,   0,   0,   7,   0,   0,   7,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,  11,   3,   1,  64, 129,
        143, 170, 204, 107,   0,   0,   1,   0,   0,   6,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,  28, 164, 254, 255, 236,
        235, 255, 249, 237,  79,   2,   6,   8,   0,   0,   0,   0,   0,
          0,   0],
       [  

In [ ]:
imgArray2=imgArray.reshape(1,-1)
imgArray2#한줄

array([[  0,   0,   0,   0,   0,   0,   0,   0,   3,   0,   0,  15,   6,
          0,   8,   0,   9,   0,   7,   0,   0,   0,   1,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  11,   2,
          0,   0,   0,  18,   0,   0,   0,   0,   0,   0,   0,   0,   5,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,  16,  19,   0,   0,   5,   2,  18,   0,  18,   3,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,  13,   0,   0,   4,   0,   0,   0,   6,   0,   0,   7,
          0,   0,   7,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,  11,   3,   1,  64, 129, 143, 170, 204, 107,   0,
          0,   1,   0,   0,   6,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,  28, 164, 254, 255, 236, 235, 255, 249,
        237,  79,   2,   6,   8,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0, 191, 177

In [59]:
#정규화 전 np=>torch
#numpy->torch
img_input=torch.tensor(imgArray2)
img_input.shape

torch.Size([1, 784])

In [60]:
#정규화=>학습
img_input=img_input/255.
img_input.shape

torch.Size([1, 784])

In [63]:
#단일 데이터의 예측함수 data_loader=>inputs
def predictOne(model,inputs,device):
    model.eval()
    predictions=[]#for문 하나짜리
    with torch.no_grad():
            inputs = inputs.to(device)
            outputs=model(inputs)
            _,predicted=outputs.max(1)
            predictions.extend(predicted.cpu().numpy())
    return predictions

In [64]:
predictOne(model,img_input,device)

[np.int64(3)]

In [ ]:
#클라스를 다양하게 변환